Weapons and Ammunition data analysis


In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

pd.options.display.max_columns = None

df = pd.read_csv('/workspaces/amc-research-sprint-lh_gl/duplicates_result.csv', encoding='latin-1')
weapons_df = df[df['item_type'] == 0]

### Select weapons life cycle stages

In [17]:
cols_to_keep = [
    # Identifiers
    'item_type', 'item',

    # Lifecycle stages - ban flags
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',

    # Lifecycle stages - restriction flags
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal'
]

weapons_lifecycle_df = weapons_df[cols_to_keep]


### Testing correlations between different life cycle stages of weapons/ammunitions

In [18]:
# Separate ban and restriction columns
ban_cols = [c for c in weapons_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in weapons_lifecycle_df.columns if 'restriction' in c]

# Correlation between every ban col vs every restriction col
corr_matrix = weapons_lifecycle_df[ban_cols + restriction_cols].corr()

# Slice to only show ban vs restriction (not ban vs ban or restriction vs restriction)
corr_ban_vs_restriction = corr_matrix.loc[ban_cols, restriction_cols]
print(corr_ban_vs_restriction)

                 restriction_development  testing_restriction  \
ban_development                      NaN            -0.033278   
ban_testing                          NaN                  NaN   
ban_production                       NaN            -0.047619   
ban_acquisition                      NaN            -0.239046   
ban_possession                       NaN                  NaN   
ban_station                          NaN                  NaN   
ban_transfer                         NaN            -0.069007   
ban_use                              NaN            -0.047619   
ban_disposal                         NaN                  NaN   

                 restriction_production  restriction_acquisition  \
ban_development               -0.126886                -0.048224   
ban_testing                         NaN                      NaN   
ban_production                -0.181568                -0.069007   
ban_acquisition                0.388217                -0.346410   
ban_posse

In [19]:
# Stack matrix into a series and sort
corr_ranked = (
    corr_ban_vs_restriction
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print(corr_ranked.head(40))

                ban              restriction  correlation
26  ban_acquisition   restriction_production     0.388217
6   ban_development          restriction_use    -0.023256
1   ban_development      testing_restriction    -0.033278
22   ban_production          restriction_use    -0.033278
62          ban_use          restriction_use    -0.033278
57          ban_use      testing_restriction    -0.047619
17   ban_production      testing_restriction    -0.047619
54     ban_transfer          restriction_use    -0.048224
3   ban_development  restriction_acquisition    -0.048224
49     ban_transfer      testing_restriction    -0.069007
19   ban_production  restriction_acquisition    -0.069007
59          ban_use  restriction_acquisition    -0.069007
51     ban_transfer  restriction_acquisition    -0.100000
5   ban_development     restriction_transfer    -0.109676
2   ban_development   restriction_production    -0.126886
61          ban_use     restriction_transfer    -0.156941
21   ban_produ

In [20]:
# Pearson - default, fine for binary
weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='pearson')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,1.000000,NaN,0.698836,0.139212,NaN,NaN,0.482243,0.698836,NaN,NaN,-0.033278,-0.126886,-0.048224,NaN,-0.109676,-0.023256,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,0.698836,NaN,1.000000,0.199205,NaN,NaN,0.690066,0.476190,NaN,NaN,-0.047619,-0.181568,-0.069007,NaN,-0.156941,-0.033278,NaN
ban_acquisition,0.139212,NaN,0.199205,1.000000,NaN,NaN,0.288675,-0.019920,NaN,NaN,-0.239046,0.388217,-0.346410,NaN,-0.306382,-0.167054,NaN
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,0.482243,NaN,0.690066,0.288675,NaN,NaN,1.000000,0.310530,NaN,NaN,-0.069007,-0.263117,-0.100000,NaN,-0.227429,-0.048224,NaN
ban_use,0.698836,NaN,0.476190,-0.019920,NaN,NaN,0.310530,1.000000,NaN,NaN,-0.047619,-0.181568,-0.069007,NaN,-0.156941,-0.033278,NaN
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Spearman - better for ordinal/binary data, more robust
weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,1.000000,NaN,0.698836,0.139212,NaN,NaN,0.482243,0.698836,NaN,NaN,-0.033278,-0.126886,-0.048224,NaN,-0.109676,-0.023256,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,0.698836,NaN,1.000000,0.199205,NaN,NaN,0.690066,0.476190,NaN,NaN,-0.047619,-0.181568,-0.069007,NaN,-0.156941,-0.033278,NaN
ban_acquisition,0.139212,NaN,0.199205,1.000000,NaN,NaN,0.288675,-0.019920,NaN,NaN,-0.239046,0.388217,-0.346410,NaN,-0.306382,-0.167054,NaN
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,0.482243,NaN,0.690066,0.288675,NaN,NaN,1.000000,0.310530,NaN,NaN,-0.069007,-0.263117,-0.100000,NaN,-0.227429,-0.048224,NaN
ban_use,0.698836,NaN,0.476190,-0.019920,NaN,NaN,0.310530,1.000000,NaN,NaN,-0.047619,-0.181568,-0.069007,NaN,-0.156941,-0.033278,NaN
ban_disposal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restriction_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Filter strong correlations

In [22]:
corr = weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

strong_corr = (
    corr
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
)

strong_corr = strong_corr[
    ((strong_corr['correlation'] > 0.5) | (strong_corr['correlation'] < -0.5)) & (strong_corr['correlation'] != 1.0)
].sort_values('correlation', ascending=False)

print(strong_corr)

                 ban      restriction  correlation
2    ban_development   ban_production     0.698836
7    ban_development          ban_use     0.698836
34    ban_production  ban_development     0.698836
119          ban_use  ban_development     0.698836
40    ban_production     ban_transfer     0.690066
104     ban_transfer   ban_production     0.690066
